<a href="https://colab.research.google.com/github/samato88/5-Plus/blob/main/SJ2025_ParseCRLData.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Takes in CRL general file with all data and does the following:
(currently working on step 5)

```
Step 1: remove all out of scope from file
Step 2: add derived pub year count. Change 9999 to 2025  and calculate
Step 3: add derived percent of publication
Step 4: map in library name and consortia from separate Participants file
Step 5: creates a new df_papr for mapping papr programs
Step 6: writes df to xlsx and df_papr to a papr tab in same xlsx
```
 Base code written by Gemini 2.5 Flash, April 2, 2026


In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')
#drive.flush_and_unmount()  # Unmount existing mount - trying since not reading new files
#drive.mount('/content/drive', force_remount=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
#@title Install and use ipdb debugger, Install xl packages
!pip install ipdb
!pip install xlsxwriter
!pip install openpyxl

import ipdb

In [3]:
#@title Other Imports (pandas, numpy, etc.)
import pandas as pd
import os
import io
import numpy as np

In [4]:
#@title Define File to Use

Test = True
path = "/content/drive/MyDrive/Colab Notebooks/Serials2025/"

if Test:
  file = 'InputFiles/TestData.xlsx'
  outputfile = 'DataOutputForTableau-TEST.xlsx'
else:
  file = 'InputFiles/CRL_General_Ouptput-20260330.txt'
  outputfile = 'DataOutputForTableau.xlsx'

symbols_filename = path + 'InputFiles/' + 'Participant_Symbols.xlsx'
input_filename = path + file
print(input_filename)

/content/drive/MyDrive/Colab Notebooks/Serials2025/InputFiles/TestData.xlsx


In [6]:
#@title Set Data Types
data_types = {
    'oclc_symbol': 'str',
    'bib_ids': 'str',
    'title': 'str',
    'local_oclc': 'str',
    'wc_oclc': 'str',
    'local_issn': 'str',
    'wc_issn': 'str',
    'lc_class': 'str',
    'language': 'str',
    'country': 'str',
    'publication_range': 'object',
    'bib_lvl': 'str',
    'form': 'str',
    'content_type': 'str',
    'media_type': 'str',
    'carrier_type':'str',
    'serial_type': 'str',
    'govt_pub': 'str',
    'east_holder_count': 'Int64',
    'east_holders': 'str',
    'east_year_range': 'str',
    'east_papr': 'Int64',
    'papr': 'Int64',
    'papr_programs': 'str',
    'papr_url': 'str',
    'crl': 'Int64',
    'crl_url': 'str',
    'hathi': 'Int64',
    'hathi_year_range': 'str',
    'portico': 'Int64',
    'wc_holder_count': 'Int64',
    'wc_holders': 'str',
    'issn_mismatch': 'Int64',
    'local_issn_not_in_issn_db': 'Int64',
    'wc_issn_not_in_issn_db': 'Int64',
    'superseded_oclc': 'Int64',
    'no_worldcat_record': 'Int64',
    'jstor': 'Int64',
    'newspaper': 'Int64',
    'monograph': 'Int64',
    'monographic_series': 'Int64',
    'other_non_serial': 'Int64',
    'not_print': 'Int64',
    'not_language_material_record_type': 'Int64',
    'no_in_scope_holdings': 'Int64',
    'out_of_scope': 'str',
    'in_scope_holdings': 'str',
    'in_scope_supplements': 'str',
    'in_scope_indexes': 'str',
    'year_range': 'str',
    'year_count': 'Int64'
    }



In [7]:
#@title Read the CSV into a pandas DataFrame
try:
    if Test:
      df = pd.read_excel(input_filename, dtype=data_types)
    else:
      df = pd.read_csv(input_filename, sep='\t', dtype=data_types)
    print("DataFrame created successfully.")
except Exception as e:
    print(f"Error reading CSV into DataFrame: {e}")
    exit()

# Display initial information about the DataFrame
#print("\n--- Initial DataFrame Info ---")
#df.info()
#print("\n--- Initial DataFrame Head ---")
#print(df.head())

DataFrame created successfully.


In [8]:
#@title Remove rows where the "out_of_scope" column is "1"

#display(df['out_of_scope'].head()) # is 1 or NaN

if 'out_of_scope' in df.columns:
    initial_rows = len(df)
    print("Initial rows:", initial_rows)
    df = df[df['out_of_scope'] != '1']
    print("Rows after removing out of scope", len(df))
    #print(f"\nRemoved {initial_rows - len(df)} rows where 'out_of_scope' was '1'.")
else:
    print("\nWarning: 'out_of_scope' column not found in the dataset. No rows were removed based on this criteria.")


Initial rows: 100
Rows after removing out of scope 85


In [9]:
#@title Create publication_year_count column
# Ensure 'publication_range' is treated as string and handle potential NaNs
df['publication_range'] = df['publication_range'].astype(str)

# Split 'publication_range' into start and end years
# Use expand=True to create new columns directly
# Apply .str.split() only to non-NaN values to prevent errors
df[['publication_range_start', 'publication_range_end']] = df['publication_range'].str.split('-', expand=True)

# Convert to Int64, coercing errors to NaN
df['publication_range_start'] = pd.to_numeric(df['publication_range_start'], errors='coerce').astype('Int64')
df['publication_range_end'] = pd.to_numeric(df['publication_range_end'], errors='coerce').astype('Int64')

# If publication_range_end is 9999, change it to 2025
df['publication_range_end'] = df['publication_range_end'].replace(9999, 2025)

# Calculate publication_year_count
df['publication_year_count'] = df['publication_range_end'] - df['publication_range_start'] + 1 # year sub off by one, add it back

#print("New columns created: 'publication_range_start', 'publication_range_end', 'publication_year_count'")
#print("First 5 rows with new columns:")
#print(df[['publication_range', 'publication_range_start', 'publication_range_end', 'publication_year_count']].head())

In [10]:
#@title Create percent_held column

# Handle division by zero and NaN values before division
# If publication_year_count is 0 or NaN, percent_held should be NaN
df['percent_held'] = np.where(
    (df['publication_year_count'].notna()) & (df['publication_year_count'] != 0),
    (df['year_count'] / df['publication_year_count'] * 100),
    np.nan
)

# Round the float values before converting to Int64 to avoid TypeError
df['percent_held'] = df['percent_held'].round().astype('Int64')

print("New column 'percent_held' created successfully.")
#print(df[['year_count', 'publication_year_count', 'percent_held']].head())

New column 'percent_held' created successfully.


In [11]:
#@title Create 'Library' and 'Consortia' columns based on 'oclc_symbol' lookup
try:
    # Load the symbols file
    symbols_df = pd.read_excel(symbols_filename)

    # Assuming the symbol is in the first column and library name in the second
    symbol_column_name = symbols_df.columns[0] # Assuming first column is symbol
    library_column_name = symbols_df.columns[1] # Assuming second column is library

    # Create a mapping dictionary from symbol to library name
    symbol_to_library_map = symbols_df.set_index(symbol_column_name)[library_column_name].to_dict()

    # Create the 'Library' column in df by mapping 'oclc_symbol'
    # Use .get() with a default value for symbols not found in the map
    df['Library'] = df['oclc_symbol'].apply(lambda x: symbol_to_library_map.get(x, 'Unknown Library'))

    print("New column 'Library' created successfully.")
    #print(df[['oclc_symbol', 'Library']].head())

    # Check if 'Consortia' column exists in symbols_df
    if 'Consortia' in symbols_df.columns:
        # Create a mapping dictionary from symbol to consortia name
        symbol_to_consortia_map = symbols_df.set_index(symbol_column_name)['Consortia'].to_dict()

        # Create the 'Consortia' column in df by mapping 'oclc_symbol'
        df['Consortia'] = df['oclc_symbol'].apply(lambda x: symbol_to_consortia_map.get(x, 'Unknown Consortia'))
        print("New column 'Consortia' created successfully.")
        #print(df[['oclc_symbol', 'Consortia']].head())
    else:
        print("Warning: 'Consortia' column not found in the symbols file. 'Consortia' column will not be created.")

except FileNotFoundError:
    print(f"Error: The symbols file '{symbols_filename}' was not found.")
    print("Please ensure the file exists in the specified path.")
except Exception as e:
    print(f"An error occurred while creating the 'Library' or 'Consortia' column: {e}")

New column 'Library' created successfully.
New column 'Consortia' created successfully.


In [12]:
#@title Create papr_df for tableau filter
papr_df = (
    df[['wc_oclc', 'papr_programs']]
    .drop_duplicates(subset=['wc_oclc'], keep='first')
    .copy()
)

# Ensure papr_programs is string type and handle NaN by filling with empty string
papr_df['papr_programs'] = papr_df['papr_programs'].astype(str).replace('nan', '')

# Split the papr_programs column by ';' and explode into new rows
papr_df['papr_programs'] = papr_df['papr_programs'].str.split(';') # split into a list of strings
papr_df = papr_df.explode('papr_programs')

# Strip whitespace from the papr_program values
papr_df['papr_programs'] = papr_df['papr_programs'].str.strip()

# Filter out empty strings that might result from splitting (e.g., if original was 'Program A;;Program B')
papr_df = papr_df[papr_df['papr_programs'] != '']

# Rename the column as requested
papr_df = papr_df.rename(columns={'papr_programs': 'papr_program'})

# Display the first few rows of the new DataFrame
print("papr_df created successfully:")
#print(papr_df.head())
print(f"Total rows in papr_df: {len(papr_df)}")

papr_df created successfully:
Total rows in papr_df: 406


In [13]:
#@title Write DataFrames to Excel

#@title Create blank 'east_institutions' column
# Create a new column named 'east_institutions' and fill it with None (which pandas handles as NaN)
df['east_institutions'] = None
print("Blank column 'east_institutions' created successfully.")
# print(df[['east_institutions']].head())

# Define the column mapping
column_mapping = {
    'oclc_symbol': 'Symbol',
    'title': 'Title',
    'wc_oclc': 'OCN',
    'wc_issn': 'ISSN',
    'lc_class': 'LC Class',
    'language': 'Language',
    'country': 'Country',
    'publication_range': 'Pub Range',
    'east_holder_count': 'Overlap',
    'east_holders': 'EAST Holders',
    'east_institutions': 'EAST Retainers',
    'east_year_range': 'Existing EAST Retention Holdings Range',
    'east_papr': 'Existing EAST Retention',
    'papr_url': 'PAPR URL',
    'crl': 'CRL Print',
    'crl_url': 'CRL URL',
    'hathi': 'HathiTrust',
    'hathi_year_range': 'HathiTrust Year Range',
    'portico': 'Portico',
    'wc_holder_count': 'WC Holdings',
    'jstor': 'JSTOR',
    'in_scope_holdings': 'In Scope Holdings',
    'in_scope_supplements': 'In Scope Supplements',
    'in_scope_indexes': 'In Scope Indexes',
    'year_range': 'Library Year Range',
    'year_count': 'Library Year Count',
    'publication_year_count': 'Publication Year Count',
    'percent_held': 'Percent Held',
    'Library': 'Library',
    'Consortia': 'Consortia'
}

# Prepare the main DataFrame (df) for output
# Select only the columns that are in the mapping and rename them
df_output = df[list(column_mapping.keys())].rename(columns=column_mapping)

# Construct the full output path
output_full_path = os.path.join(path, outputfile)

print(f"Writing output to: {output_full_path}")

try:
    with pd.ExcelWriter(output_full_path, engine='xlsxwriter') as writer:
        # Write df_output to the first sheet
        df_output.to_excel(writer, sheet_name='Data', index=False)

        # Write papr_df to a second sheet
        papr_df.to_excel(writer, sheet_name='papr', index=False)

    print("DataFrames successfully written to Excel file.")
except Exception as e:
    print(f"Error writing DataFrames to Excel: {e}")

Blank column 'east_institutions' created successfully.
Writing output to: /content/drive/MyDrive/Colab Notebooks/Serials2025/DataOutputForTableau-TEST.xlsx
DataFrames successfully written to Excel file.
